<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Deep Learning Basics with PyTorch

**Dr. Yves J. Hilpisch with GPT-5**

# Part IV — Toward Large Language Models

## Chapter 15 — Training Large Models (DDP, AMP, Checkpointing)

## Overview

This Colab-ready notebook mirrors Chapter 15 and walks through the key techniques
for scaling PyTorch training beyond a single GPU:

- **§15.1** Why distributed? Throughput and wall-clock motivation
- **§15.2** Effective batch size and gradient accumulation (`B_global = B_micro × K × R`)
- **§15.3** AMP + GradScaler — mixed precision with safe loss scaling
- **§15.4** DDP launch sketch — `torchrun` command and `train_ddp.py` template
- **§15.5** Throughput benchmarking — samples/sec vs batch size
- **§15.6** Robust checkpointing — full state round-trip including RNG
- **§15.7** Multi-node sketch (reference)
- **§15.8** Memory-saving techniques preview (activation checkpointing, ZeRO)
- **§15.9** Common pitfalls

All executable cells run on **CPU** — no GPU required to follow along.
AMP cells detect CUDA and either demonstrate the pattern or explain the skip.

**Reuses from earlier chapters:** `TinyMLP` pattern (Ch 8/12),
AMP + GradScaler (Ch 12 §12.2), gradient accumulation (Ch 12 §12.3),
checkpoint save/load (Ch 12 §12.5), DataLoader / TensorDataset (Ch 9).

Run cells top-to-bottom; restart kernel if state gets confusing.

## Environment Check

Confirm Python version, PyTorch version, and the available compute device.

In [ ]:
import math
import os
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print(f'Python : {sys.version.split()[0]}')       # interpreter version
print(f'PyTorch: {torch.__version__}')            # torch version
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')                       # compute target
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

## Shared Fixtures

One synthetic regression dataset and one `TinyMLP` model are reused across all sections
so each demo starts from a clean, known state.

In [ ]:
torch.manual_seed(42)          # reproducibility

N, D = 512, 32                 # samples, input features
X = torch.randn(N, D)          # synthetic inputs
y = torch.randn(N, 1)          # synthetic targets

dataset = TensorDataset(X, y)  # wrap for DataLoader


def make_model(hidden: int = 128) -> nn.Module:
    """Return a fresh TinyMLP — same pattern used in Ch 8/12."""
    return nn.Sequential(
        nn.Linear(D, hidden),
        nn.ReLU(),
        nn.Linear(hidden, 1),
    ).to(device)


print(f'Dataset : {N} samples × {D} features')
print(f'TinyMLP : {D} → 128 → 1')

---
## §15.1 — Why Distributed? Throughput and Wall-Clock

Large models and datasets benefit from parallelism: multiple GPUs process different
micro-batches in lock-step, synchronising gradients each step.
Done well, this brings **near-linear speedups** until communication overhead dominates.

The table below shows the mental model for a single-node, 4-GPU DDP run:

| Rank | GPU | Micro-batch | All-reduce |
|------|-----|-------------|------------|
| 0    | 0   | rows 0–127  | ← sync →   |
| 1    | 1   | rows 128–255| ← sync →   |
| 2    | 2   | rows 256–383| ← sync →   |
| 3    | 3   | rows 384–511| ← sync →   |

On a single device the benefit is purely from AMP and a larger effective batch;
the DDP topology is a reference for when you move to multi-GPU hardware.

In [ ]:
# Baseline timing: simple FP32 training loop (1 epoch, batch=32)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
model  = make_model()
opt    = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
t0 = time.perf_counter()
for xb, yb in loader:
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(set_to_none=True)          # clear gradients
    loss = F.mse_loss(model(xb), yb)         # forward
    loss.backward()                          # backward
    opt.step()                               # update
elapsed = time.perf_counter() - t0

samples_per_sec = N / elapsed
print(f'FP32 baseline : {elapsed*1000:.1f} ms  |  '
      f'{samples_per_sec:.0f} samples/sec')

---
## §15.2 — Effective Batch Size and Gradient Accumulation

When a single device cannot hold the desired batch, accumulate gradients over
**K** micro-batches before stepping the optimizer:

$$B_{\text{global}} = B_{\text{micro}} \times K \times R$$

where `R` is the number of replicas (GPUs). On a single device `R = 1`.

**Key rule:** divide the loss by `K` before `.backward()` so the
accumulated gradient is equivalent to one large-batch gradient.

The cell below sweeps `K ∈ {1, 2, 4, 8}` and measures wall-clock per step.

In [ ]:
import matplotlib
matplotlib.use('Agg')                        # headless backend
import matplotlib.pyplot as plt

MICRO = 16                                   # micro-batch size
ACCUM_STEPS = [1, 2, 4, 8]                  # accumulation factors to sweep

times, eff_batches = [], []

for K in ACCUM_STEPS:
    model_k = make_model()
    opt_k   = torch.optim.AdamW(model_k.parameters(), lr=1e-3)
    loader_k = DataLoader(dataset, batch_size=MICRO, shuffle=False)
    model_k.train()

    t0 = time.perf_counter()
    opt_k.zero_grad(set_to_none=True)        # start accumulation window
    for step, (xb, yb) in enumerate(loader_k):
        xb, yb = xb.to(device), yb.to(device)
        loss = F.mse_loss(model_k(xb), yb) / K   # scale loss
        loss.backward()                           # accumulate
        if (step + 1) % K == 0:
            opt_k.step()                          # update every K steps
            opt_k.zero_grad(set_to_none=True)
    elapsed_k = time.perf_counter() - t0

    B_global = MICRO * K * 1              # R=1 on single device
    times.append(elapsed_k * 1000)
    eff_batches.append(B_global)
    print(f'K={K:2d}  B_global={B_global:4d}  time={elapsed_k*1000:.1f} ms')

# Plot time vs accumulation factor
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(ACCUM_STEPS, times, marker='o')
ax.set_xlabel('Accumulation steps K')
ax.set_ylabel('Time per epoch (ms)')
ax.set_title('Wall-clock vs gradient accumulation (CPU)')
ax.set_xticks(ACCUM_STEPS)
fig.tight_layout()
plt.show()
print('B_global values:', eff_batches)

### Verify gradient equivalence

Accumulating K=4 micro-batches of size 16 must produce the same weight update as
a single batch of size 64.  We confirm this numerically (max parameter delta ≈ 0).

In [ ]:
torch.manual_seed(0)
K_verify  = 4
B_micro_v = 16

big_X = torch.randn(B_micro_v * K_verify, D, device=device)
big_y = torch.randn(B_micro_v * K_verify, 1, device=device)

# ── Reference: single large batch ───────────────────────────────────
ref_model = make_model()
ref_opt   = torch.optim.SGD(ref_model.parameters(), lr=0.1)
ref_opt.zero_grad(set_to_none=True)
F.mse_loss(ref_model(big_X), big_y).backward()
ref_opt.step()
ref_params = {k: v.detach().clone() for k, v in ref_model.state_dict().items()}

# ── Accumulation: K micro-batches ────────────────────────────────────
acc_model = make_model()
# start from same weights as ref_model
acc_model.load_state_dict({k: v.clone() for k, v in ref_model.state_dict().items()})
# reset to pre-step weights
acc_model.load_state_dict(
    {k: v.clone() for k, v in
     dict(zip(ref_model.state_dict(), ref_params.values())).items()}
)
# Re-init fresh so weights match pre-step ref
torch.manual_seed(0)
acc_model = make_model()          # same init seed → same weights
torch.manual_seed(0)
ref_model = make_model()          # reset ref to same init

ref_opt = torch.optim.SGD(ref_model.parameters(), lr=0.1)
ref_opt.zero_grad(set_to_none=True)
F.mse_loss(ref_model(big_X), big_y).backward()
ref_opt.step()

acc_opt = torch.optim.SGD(acc_model.parameters(), lr=0.1)
acc_opt.zero_grad(set_to_none=True)
for chunk_x, chunk_y in zip(
        big_X.chunk(K_verify), big_y.chunk(K_verify)):
    (F.mse_loss(acc_model(chunk_x), chunk_y) / K_verify).backward()
acc_opt.step()

max_delta = max(
    (acc_model.state_dict()[k] - ref_model.state_dict()[k]).abs().max().item()
    for k in acc_model.state_dict()
)
print(f'Max parameter delta (accum vs large-batch): {max_delta:.2e}')  # expect ~0

---
## §15.3 — AMP and Loss Scaling

**Automatic Mixed Precision (AMP)** speeds up math on modern GPUs by computing
the forward and backward passes in FP16 while keeping the master weights in FP32.

- `torch.amp.autocast('cuda')` — selects FP16/BF16 ops automatically
- `GradScaler` — multiplies the loss by a large scale factor before `.backward()`
  to prevent FP16 underflow; detects overflow and **skips** the optimizer step,
  then halves the scale

The pattern below is the standard production template
(identical to Ch 12 §12.2, extended with accumulation):

In [ ]:
use_amp = torch.cuda.is_available()           # AMP requires CUDA

if use_amp:
    amp_model  = make_model()
    amp_opt    = torch.optim.AdamW(amp_model.parameters(), lr=1e-3)
    scaler     = torch.amp.GradScaler('cuda') # manages loss scale
    amp_loader = DataLoader(dataset, batch_size=32, shuffle=True)
    K_amp      = 4                             # accumulation inside AMP

    amp_model.train()
    amp_opt.zero_grad(set_to_none=True)
    losses = []

    for step, (xb, yb) in enumerate(amp_loader):
        xb, yb = xb.cuda(), yb.cuda()
        with torch.amp.autocast('cuda'):       # mixed-precision region
            loss = F.mse_loss(amp_model(xb), yb) / K_amp
        scaler.scale(loss).backward()          # scaled backward
        if (step + 1) % K_amp == 0:
            scaler.step(amp_opt)               # unscale + step (skipped on overflow)
            scaler.update()                    # adjust scale factor
            amp_opt.zero_grad(set_to_none=True)
        losses.append(loss.item() * K_amp)     # unscale for logging

    print(f'AMP run complete  |  final loss: {losses[-1]:.4f}')
    print(f'GradScaler scale : {scaler.get_scale()}')
else:
    print('CUDA not available — AMP demo skipped.')
    print('On a CUDA machine this block runs the AMP + accumulation loop.')
    print('CPU fallback: use torch.amp.autocast("cpu") for BF16 on supported CPUs.')

---
## §15.4 — Launching DDP (Single Node)

DDP is launched via the CLI with `torchrun` (PyTorch ≥ 1.9).
Each process owns **one GPU** and runs an identical copy of the script.
Gradients are all-reduced after every backward pass.

### CLI command (4 GPUs, single node)

```bash
torchrun --standalone --nproc_per_node=4 train_ddp.py \
    --batch 128 --accum 2 --amp --epochs 5
```

### Minimal `train_ddp.py` template

Save the code below as `train_ddp.py` and launch with the command above.
The cell is a reference string — it is not executed here.

In [ ]:
ddp_template = '''
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler


def setup_ddp():
    """Initialise the process group (NCCL backend on GPU)."""
    dist.init_process_group("nccl")                      # env vars set by torchrun
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))


def cleanup_ddp():
    dist.destroy_process_group()


def main():
    setup_ddp()
    rank       = dist.get_rank()           # global rank
    local_rank = int(os.environ["LOCAL_RANK"])

    # --- model -------------------------------------------------------
    model = YourModel().cuda(local_rank)   # move to this rank's GPU
    model = DDP(model, device_ids=[local_rank])  # wrap once

    # --- data --------------------------------------------------------
    sampler = DistributedSampler(dataset, shuffle=True)
    loader  = DataLoader(dataset, batch_size=128, sampler=sampler)

    # --- training loop -----------------------------------------------
    scaler = torch.amp.GradScaler("cuda")
    for epoch in range(args.epochs):
        sampler.set_epoch(epoch)           # re-shuffle per epoch
        for xb, yb in loader:
            xb, yb = xb.cuda(), yb.cuda()
            with torch.amp.autocast("cuda"):
                loss = loss_fn(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # --- checkpoint (rank 0 only) --------------------------------
        if rank == 0:
            torch.save({
                "model": model.module.state_dict(),  # unwrap DDP
                "opt"  : optimizer.state_dict(),
                "epoch": epoch,
            }, f"ckpt_epoch{epoch}.pt")
        dist.barrier()                     # all ranks wait for rank 0

    cleanup_ddp()


if __name__ == "__main__":
    main()
'''

print(ddp_template)    # display template for reference

---
## §15.5 — Throughput Benchmarking

Before adding GPUs, saturate the single device first.
Profile per stage: **data loading → forward → backward → optimizer**.

The cell below sweeps batch sizes and plots samples/sec,
illustrating the elbow beyond which memory or compute saturate.

In [ ]:
BATCH_SIZES = [8, 16, 32, 64, 128]
throughputs = []

for bs in BATCH_SIZES:
    m   = make_model()
    o   = torch.optim.AdamW(m.parameters(), lr=1e-3)
    ld  = DataLoader(dataset, batch_size=bs, shuffle=False)
    m.train()

    # warm-up
    for xb, yb in ld:
        xb, yb = xb.to(device), yb.to(device)
        o.zero_grad(set_to_none=True)
        F.mse_loss(m(xb), yb).backward()
        o.step()
        break

    t0 = time.perf_counter()
    steps = 0
    for xb, yb in ld:
        xb, yb = xb.to(device), yb.to(device)
        o.zero_grad(set_to_none=True)
        F.mse_loss(m(xb), yb).backward()
        o.step()
        steps += xb.size(0)
    elapsed = time.perf_counter() - t0

    sps = steps / elapsed
    throughputs.append(sps)
    print(f'batch={bs:4d}  throughput={sps:7.0f} samples/sec')

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(BATCH_SIZES, throughputs, marker='s', color='steelblue')
ax.set_xlabel('Batch size')
ax.set_ylabel('Samples / sec')
ax.set_title('Throughput vs batch size (CPU)')
ax.set_xticks(BATCH_SIZES)
fig.tight_layout()
plt.show()

---
## §15.6 — Robust Checkpointing

Save **everything needed for a clean resume**:

| Key | Contents |
|-----|----------|
| `model` | `model.state_dict()` (use `model.module.state_dict()` in DDP) |
| `opt`   | `optimizer.state_dict()` — preserves momentum, AdamW second moments |
| `sched` | `scheduler.state_dict()` — preserves LR schedule position |
| `epoch` | current epoch number for resume logic |
| `rng`   | `torch.get_rng_state()` — exact data-order reproducibility |

**DDP rule:** only rank 0 writes; all ranks call `dist.barrier()` afterwards.

In [ ]:
import tempfile, pathlib

# ── Train for 3 steps to produce non-trivial state ──────────────────
ckpt_model = make_model()
ckpt_opt   = torch.optim.AdamW(ckpt_model.parameters(), lr=1e-3)
ckpt_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    ckpt_opt, T_max=10)
ckpt_loader = DataLoader(dataset, batch_size=32, shuffle=True)

ckpt_model.train()
for epoch in range(3):
    for xb, yb in ckpt_loader:
        xb, yb = xb.to(device), yb.to(device)
        ckpt_opt.zero_grad(set_to_none=True)
        F.mse_loss(ckpt_model(xb), yb).backward()
        ckpt_opt.step()
    ckpt_sched.step()

# ── Save checkpoint ─────────────────────────────────────────────────
ckpt_dir  = pathlib.Path(tempfile.mkdtemp())
ckpt_path = ckpt_dir / 'ch15_demo.pt'

state = {
    'model' : ckpt_model.state_dict(),
    'opt'   : ckpt_opt.state_dict(),
    'sched' : ckpt_sched.state_dict(),
    'epoch' : 3,
    'rng'   : torch.get_rng_state(),     # CPU RNG state
}
torch.save(state, ckpt_path)
print(f'Saved  → {ckpt_path}  ({ckpt_path.stat().st_size / 1024:.1f} KB)')

# ── Load and verify round-trip ───────────────────────────────────────
loaded = torch.load(ckpt_path, map_location=device, weights_only=False)

resume_model = make_model()
resume_opt   = torch.optim.AdamW(resume_model.parameters(), lr=1e-3)
resume_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    resume_opt, T_max=10)

resume_model.load_state_dict(loaded['model'])
resume_opt.load_state_dict(loaded['opt'])
resume_sched.load_state_dict(loaded['sched'])
torch.set_rng_state(loaded['rng'])

# Confirm weights identical after round-trip
max_diff = max(
    (resume_model.state_dict()[k] - ckpt_model.state_dict()[k]).abs().max().item()
    for k in ckpt_model.state_dict()
)
print(f'Resumed from epoch : {loaded["epoch"]}')
print(f'LR after resume    : {resume_sched.get_last_lr()}')
print(f'Max weight delta   : {max_diff:.2e}  (expect 0.00e+00)')

---
## §15.7 — Multi-Node Sketch

Scale beyond a single machine by specifying `--nnodes` and a shared rendezvous endpoint.
Run this command on **every node** (with identical `--node-rank`):

```bash
# Node 0  (the rendezvous host)
torchrun --nnodes=2 --nproc_per_node=4 \
         --rdzv_backend=c10d --rdzv_endpoint=HOST0:29400 \
         --node_rank=0 \
         train_ddp.py --batch 128 --amp

# Node 1
torchrun --nnodes=2 --nproc_per_node=4 \
         --rdzv_backend=c10d --rdzv_endpoint=HOST0:29400 \
         --node_rank=1 \
         train_ddp.py --batch 128 --amp
```

The `c10d` rendezvous backend uses a TCP store on `HOST0:29400`.
Both nodes must have network access to each other and the same code + environment.

---
## §15.8 — Memory-Saving Techniques (Preview)

When model or optimizer state does not fit in GPU memory, two main strategies help:

### Activation Checkpointing

Discard intermediate activations during the forward pass; recompute them during
the backward pass.  Trades compute for memory.

```python
from torch.utils.checkpoint import checkpoint

# Replace a block's forward call:
out = checkpoint(block, x)   # recomputes block(x) during backward
```

### ZeRO-Style Sharding

Partition **parameters**, **optimizer state**, and **gradients** across ranks;
all-gather only the slice needed for each forward/backward.
PyTorch's `FullyShardedDataParallel` (FSDP) implements this.

| Stage | What is sharded |
|-------|----------------|
| ZeRO-1 | Optimizer state |
| ZeRO-2 | + Gradients |
| ZeRO-3 | + Parameters |

```python
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
model = FSDP(model)   # drop-in replacement for DDP at ZeRO-3 level
```

---
## §15.9 — Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Loss not divided by K | Gradients K× too large | Divide loss by `accum_steps` before `.backward()` |
| Tiny micro-batches | Low GPU utilisation | Raise micro-batch until memory is full |
| CPU input stalls | GPU idle between steps | Increase `DataLoader(num_workers=…, pin_memory=True)` |
| Oversized global batch | Generalisation degrades | Scale LR with batch size (linear scaling rule) |
| AMP divergence | NaN loss | Check `GradScaler` scale; lower LR; verify BN layers |
| Only saving model | Cannot resume training | Always save optimizer + scheduler + epoch + RNG |
| All ranks write checkpoints | File conflicts in DDP | Gate writes with `if dist.get_rank() == 0:` |
| DistributedSampler without `set_epoch` | Same order every epoch | Call `sampler.set_epoch(epoch)` before each loop |

---
## Summary and Quick Checks

**Core formulas**
- $B_{\text{global}} = B_{\text{micro}} \times K \times R$
- Linear LR scaling rule: $\text{lr}_{\text{new}} = \text{lr}_{\text{base}} \times B_{\text{global}} / B_{\text{ref}}$

**Quick checks before scaling out**
- [ ] AMP + GradScaler trains stably (no NaN); fall back to FP32 on CPU
- [ ] Loss is divided by `accum_steps` inside the accumulation loop
- [ ] Checkpoint round-trip works on both CPU and GPU (`map_location`)
- [ ] `DistributedSampler.set_epoch(epoch)` called each epoch
- [ ] Only rank 0 writes checkpoints; barrier called afterwards

**Where we're heading next**

Chapter 16 steps back to **ethics, risks, and responsible deployment** — model cards,
fairness metrics, privacy, and guardrails.

See scripts and figure generators in Appendix F.  
Companion notebook: `ch15_training_large_models.ipynb`.

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>